# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [ ]:
# !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [ ]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [ ]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [ ]:
doc_path =  "./datasets/state_of_the_union.txt"

In [ ]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [ ]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [ ]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [ ]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [ ]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [ ]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

In [ ]:
agent.invoke("Why use ruff over flake8?")

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [ ]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [ ]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

In [ ]:
agent.invoke("Why use ruff over flake8?")

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [ ]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [ ]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



### 1. Get your data

For this replication we're swapping in Shakespeare's *sonnets.txt* (linked above). It's downloaded straight from the public GitHub repo and saved into the local `datasets/` folder, right next to `state_of_the_union.txt`, so it can be loaded the exact same way with `TextLoader`.

In [ ]:
import os
import urllib.request

sonnets_url = "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"
sonnets_path = "./datasets/sonnets.txt"

if not os.path.exists(sonnets_path):
    urllib.request.urlretrieve(sonnets_url, sonnets_path)

print(f"Dataset ready at {sonnets_path}")

### 2. Rebuild the vector store

Same recipe as the state of the union speech: load the raw text, split it into chunks, embed the chunks, and store them in their own Chroma collection so they don't mix with the other two knowledge sources. Then wrap the resulting retriever in a fresh `RetrievalQA` chain.

In [ ]:
sonnets_loader = TextLoader(sonnets_path)
sonnets_documents = sonnets_loader.load()
sonnets_texts = text_splitter.split_documents(sonnets_documents)

sonnets_db = Chroma.from_documents(sonnets_texts, embeddings, collection_name="sonnets")

sonnets_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)

### 3 & 4. Rewrite the tool description and rebuild the agent

We drop the state-of-the-union tool and pair the new Sonnets tool with the existing Ruff tool, so the agent still has two very different knowledge sources to route between. The `description` is the only thing the agent sees when deciding which tool to call, so it needs to clearly signal *when* this tool is the right pick.

In [ ]:
tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,
        description="useful for when you need to answer questions about Shakespeare's sonnets, such as their themes, imagery, language, or the content of specific sonnets. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

### 5. Put it to the test

First, a question only the Sonnets tool can answer.

In [ ]:
agent.invoke("What is the central theme of Shakespeare's Sonnet 18?")

Next, a question only the Ruff tool can answer.

In [ ]:
agent.invoke("Why use ruff over flake8?")

Finally, a multi-hop question that requires combining knowledge from *both* tools, mirroring the ruff/Jupyter example above.

In [ ]:
agent.invoke(
    "Ruff can flag lines that are too long. Does Shakespeare's Sonnet 18 stick to a strict line length, and what is that structure called?"
)

Let's also rebuild the router version (`return_direct=True`) with the new tools, so we can compare its behavior to the reasoning agent above.

In [ ]:
tools_direct = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,
        description="useful for when you need to answer questions about Shakespeare's sonnets, such as their themes, imagery, language, or the content of specific sonnets. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

agent_direct = initialize_agent(
    tools_direct, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

agent_direct.invoke("What is the central theme of Shakespeare's Sonnet 18?")

### 6. Reflect

**Did the agent pick the right tool(s)?** Yes. For the Sonnet 18 question, the tool description's mention of "Shakespeare's sonnets" lines up directly with the wording of the question, so the agent's `Action` step chose the Shakespeare Sonnets QA System. For the "ruff vs flake8" question, the vocabulary ("ruff", "linter"-adjacent phrasing) matches the Ruff tool's description just as unambiguously, so that tool was picked instead. Because the two descriptions describe non-overlapping topics, the agent doesn't have to guess — it just pattern-matches the question against whichever description is the closest fit. For the multi-hop question, the reasoning agent (`return_direct=False`) needs two steps: it first calls the Ruff tool to confirm that line-length is something Ruff can flag, then calls the Sonnets tool to check Sonnet 18's structure, and finally combines both observations ("yes, Ruff flags long lines; yes, sonnets follow a strict 14-line, iambic-pentameter structure") into one synthesized answer.

**`return_direct=True` vs. not:** With `return_direct=False` (the default), the agent can chain multiple tool calls and reason over their combined output before producing a final answer — this is what makes the multi-hop question answerable at all. With `return_direct=True`, the agent stops as soon as the first tool call returns: whatever that `RetrievalQA` chain outputs is returned verbatim as the final answer, with no further `Thought`/`Action` steps. That's perfect for simple single-source lookups (faster, cheaper, no risk of the agent "rambling" after a good answer), but it breaks multi-hop questions, since the agent never gets a chance to call the second tool.

*Note: this notebook was completed without executing the cells, so the reasoning above is a best-effort prediction of the agent's behavior based on how `ZERO_SHOT_REACT_DESCRIPTION` and `return_direct` work, not a transcript of an actual run.*